# Nodule finding exam (GPU) — find, measure, match on RIDER

**What this does:** runs the engine's nodule finder (free TotalSegmentator *lung_nodules* model + lobes) on all
59 expert-outlined RIDER scans, measures each nodule the way Daniel does (long + perpendicular axis on the
largest slice, volume from the outline), pairs nodules between the test and retest scan, and checks:
1. was the expert-outlined tumour found? 2. how close are the measurements? 3. was the tumour paired correctly?
4. how many other candidates appear, and are they seen in both scans (15 min apart)?

**How to run:** `Runtime → Change runtime type → T4 GPU`, then `Runtime → Run all` (≈ 30–45 min, mostly downloads).
At the end two files download: `finding_results.zip` (CSVs) and `engine_cache.zip` (AI masks, so the free server
doesn't need to recompute). *Research use only. Public, de-identified data.*

In [ ]:
N_CASES = 59   # use e.g. 6 for a quick first run
import os; os.environ["ENGINE_SPLIT_LUNGS"] = "0"   # GPU has the memory to see both lungs at once
import torch; print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - switch to T4 GPU!")
!pip -q install totalsegmentator pydicom SimpleITK scikit-image pandas
!totalseg_download_weights -t lung_nodules > /dev/null && totalseg_download_weights -t total_fast > /dev/null && echo weights ok

## 1. Engine code (same files as `engine/` in the repo)

In [ ]:
!mkdir -p /content/engine /content/cache

In [ ]:
%%writefile /content/engine/__init__.py


In [ ]:
%%writefile /content/engine/detect.py
"""Nodule finding + lobes, with two free, open models (TotalSegmentator family, nnU-Net).

Step 1  lobes: TotalSegmentator 'total' 3 mm model (5 lobe labels used).
Step 2  nodules: TotalSegmentator 'lung_nodules' model (nnU-Net ResEnc, trained on 1352 CTs by
        BlueMind AI, no licence needed), run on the lung crop at the scan's own resolution.

Both models are called directly through nnU-Net in ONE process (no worker processes): the free CPU
machines we may use have 6-16 GB RAM, and TotalSegmentator's default runner needs more.
Everything returns arrays on the original CT grid (z, y, x), so measurements stay in CT space.
"""
import os, tempfile
import numpy as np
import SimpleITK as sitk
import torch

TS_HOME = os.path.expanduser(os.environ.get("TOTALSEG_HOME_DIR", "~/.totalsegmentator"))
LOBE_MODEL = f"{TS_HOME}/nnunet/results/Dataset297_TotalSegmentator_total_3mm_1559subj/nnUNetTrainer_4000epochs_NoMirroring__nnUNetPlans__3d_fullres"
NODULE_MODEL = f"{TS_HOME}/nnunet/results/Dataset913_lung_nodules/nnUNetTrainer_MOSAIC_1k_QuarterLR_NoMirroring__nnUNetPlans__3d_fullres"
# total-model label -> our lobe code index
LOBE_LABELS = {12: 1, 13: 2, 14: 3, 10: 4, 11: 5}
LOBE_CODE = {1: "RUL", 2: "RML", 3: "RLL", 4: "LUL", 5: "LLL"}

_predictors = {}


def _threads():
    torch.set_num_threads(int(os.environ.get("ENGINE_THREADS", os.cpu_count() or 2)))


def _device():
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


def _predictor(model_dir, patch=None, step=0.5):
    key = (model_dir, patch, step)
    if key not in _predictors:
        import totalsegmentator.nnunet  # noqa: F401  registers TotalSegmentator's trainer classes
        from nnunetv2.inference.predict_from_raw_data import nnUNetPredictor
        dev = _device()
        p = nnUNetPredictor(tile_step_size=step, use_gaussian=True, use_mirroring=False,
                            perform_everything_on_device=dev.type == "cuda", device=dev, allow_tqdm=False)
        folds = ("0",) if os.path.isdir(f"{model_dir}/fold_0") else ("all",)
        p.initialize_from_trained_model_folder(model_dir, use_folds=folds)
        if patch:
            p.configuration_manager.configuration["patch_size"] = list(patch)
        if dev.type == "cpu" and os.environ.get("ENGINE_CPU_BF16", "0") == "1":
            # bfloat16 on CPU halves activation memory BUT degrades this model badly (tested 15 Oct:
            # mean tumour probability 0.80 -> 0.37). Off by default; kept only for experiments.
            net = p.network
            fwd = net.forward
            def forward_bf16(x, _fwd=fwd):
                with torch.autocast("cpu", dtype=torch.bfloat16):
                    return _fwd(x).float()
            net.forward = forward_bf16
        _predictors[key] = p
    return _predictors[key]


def _run(model_dir, img, patch=None, step=0.5, prob_label=None):
    """Run an nnU-Net model on a SimpleITK image. Returns (label array, prob array or None) on img's grid."""
    _threads()
    pred = _predictor(model_dir, patch, step)
    rw = pred.plans_manager.image_reader_writer_class()  # same reader as in training (reorients)
    with tempfile.TemporaryDirectory() as d:
        sitk.WriteImage(img, f"{d}/x_0000.nii.gz")
        arr, props = rw.read_images([f"{d}/x_0000.nii.gz"])
        with torch.inference_mode():
            out = pred.predict_single_npy_array(arr, props, None, None, prob_label is not None)
        seg, prob = out if prob_label is not None else (out, None)
        rw.write_seg(seg.astype(np.uint8), f"{d}/seg.nii.gz", props)
        seg = sitk.GetArrayFromImage(sitk.ReadImage(f"{d}/seg.nii.gz"))
        if prob is not None:
            rw.write_seg(np.round(prob[prob_label] * 1000).astype(np.int16), f"{d}/p.nii.gz", props)
            prob = sitk.GetArrayFromImage(sitk.ReadImage(f"{d}/p.nii.gz")).astype(np.float32) / 1000
    return seg, prob


def lobes(ct):
    """Lobe label map (0 = outside, 1..5 = RUL, RML, RLL, LUL, LLL) on the CT grid. ct: path or image."""
    ref = sitk.ReadImage(ct) if isinstance(ct, str) else ct
    sp3 = [3.0, 3.0, 3.0]
    size3 = [int(round(s * p / 3.0)) for s, p in zip(ref.GetSize(), ref.GetSpacing())]
    small = sitk.Resample(ref, size3, sitk.Transform(), sitk.sitkLinear, ref.GetOrigin(), sp3,
                          ref.GetDirection(), -1024, sitk.sitkInt16)
    seg, _ = _run(LOBE_MODEL, small)
    release(LOBE_MODEL)
    lab3 = np.zeros(seg.shape, np.uint8)
    for k, v in LOBE_LABELS.items():
        lab3[seg == k] = v
    li = sitk.GetImageFromArray(lab3); li.CopyInformation(small)
    return sitk.GetArrayFromImage(sitk.Resample(li, ref, sitk.Transform(), sitk.sitkNearestNeighbor, 0, sitk.sitkUInt8))


def release(model_dir):
    """Drop a cached model to free memory (the lobe model is not needed once lobes exist)."""
    import gc, ctypes
    for k in [k for k in _predictors if k[0] == model_dir]:
        del _predictors[k]
    gc.collect()
    try:
        ctypes.CDLL("libc.so.6").malloc_trim(0)
    except Exception:
        pass


def lung_crop(lobe_map, spacing_zyx, margin_mm=10):
    zz, yy, xx = np.nonzero(lobe_map)
    mg = np.ceil(margin_mm / np.asarray(spacing_zyx)).astype(int)
    lo = np.maximum([zz.min(), yy.min(), xx.min()] - mg, 0)
    hi = np.minimum(np.array([zz.max(), yy.max(), xx.max()]) + mg + 1, lobe_map.shape)
    return lo, hi


def nodules(ct, lobe_map, patch=None, step=0.5):
    """Nodule mask (bool) and nodule probability (float16) on the full CT grid."""
    ref = sitk.ReadImage(ct) if isinstance(ct, str) else ct
    sp = np.array(ref.GetSpacing())[::-1]
    m = np.zeros(lobe_map.shape, bool)
    p = np.zeros(lobe_map.shape, np.float16)
    # Default: both lungs in one crop (as TotalSegmentator runs it). ENGINE_SPLIT_LUNGS=1 runs one lung
    # at a time to save memory, but the model then sees less context (untested for accuracy).
    parts = [np.isin(lobe_map, (1, 2, 3)), np.isin(lobe_map, (4, 5))]
    if os.environ.get("ENGINE_SPLIT_LUNGS", "0") == "0" or not all(x.any() for x in parts):
        parts = [lobe_map > 0]
    for part in parts:
        lo, hi = lung_crop(part, sp, margin_mm=15)
        crop = ref[int(lo[2]):int(hi[2]), int(lo[1]):int(hi[1]), int(lo[0]):int(hi[0])]
        seg, prob = _run(NODULE_MODEL, crop, patch, step, prob_label=2)
        sl = tuple(slice(int(a), int(b)) for a, b in zip(lo, hi))
        m[sl] |= seg == 2
        p[sl] = np.maximum(p[sl], prob.astype(np.float16))
        del seg, prob
    return m, p


In [ ]:
%%writefile /content/engine/measure.py
"""Measure nodules the way Daniel does.

- Long axis: longest edge-to-edge diameter on the axial slice where the nodule is largest.
- Short axis: longest diameter perpendicular to the long axis, on that same slice.
- Mean diameter = (long + short) / 2   (his sheet's formula).
- Volume (mm3) from the 3D outline, for every nodule (today he only has it for ~1 in 5).
- Lobe from the lobe map; Type (Solid / Part-solid / Ground-glass) from CT density - a draft he confirms.
Distances use pixel corners (edge to edge, like a caliper), not pixel centres.
"""
import numpy as np
from scipy import ndimage as ndi
from scipy.spatial import ConvexHull

SOLID_HU = -300  # voxels denser than this count as solid tissue


def _corners(ys, xs, sy, sx):
    c = []
    for dy in (-0.5, 0.5):
        for dx in (-0.5, 0.5):
            c.append(np.stack([(ys + dy) * sy, (xs + dx) * sx], 1))
    return np.concatenate(c)


def axes_2d(m2d, sy, sx):
    """Long axis, short axis (mm) and the two segments' endpoints [(y,x) px] for one slice."""
    ys, xs = np.nonzero(m2d)
    if len(ys) == 0:
        return 0.0, 0.0, None, None
    pts = _corners(ys.astype(float), xs.astype(float), sy, sx)
    try:
        hull = pts[ConvexHull(pts).vertices]
    except Exception:
        hull = pts
    d = np.sqrt(((hull[:, None] - hull[None]) ** 2).sum(-1))
    i, j = np.unravel_index(d.argmax(), d.shape)
    long_mm = float(d[i, j])
    u = (hull[j] - hull[i]) / max(long_mm, 1e-6)
    v = np.array([-u[1], u[0]])
    # short axis: longest chord perpendicular to u. Bin pixels by position along u.
    cen = np.stack([ys * sy, xs * sx], 1).astype(float)
    a, b = cen @ u, cen @ v
    w = min(sy, sx)
    bins = np.floor((a - a.min()) / w).astype(int)
    half = 0.5 * (abs(v[0]) * sy + abs(v[1]) * sx)  # pixel half-extent along v
    best, seg = 0.0, None
    for k in np.unique(bins):
        sel = bins == k
        lo, hi = b[sel].min() - half, b[sel].max() + half
        if hi - lo > best:
            best = hi - lo
            am = a[sel].mean()
            p0, p1 = am * u + lo * v, am * u + hi * v
            seg = (p0 / [sy, sx], p1 / [sy, sx])
    long_seg = (hull[i] / [sy, sx], hull[j] / [sy, sx])
    return long_mm, float(best), long_seg, seg


def measure_nodule(mask, ct_hu, spacing_zyx, lobe_map=None, instance_numbers=None):
    """mask: bool (z,y,x) for ONE nodule. Returns the sheet fields + geometry for the viewer."""
    sz, sy, sx = spacing_zyx
    zs = np.nonzero(mask.any(axis=(1, 2)))[0]
    best = (-1, 0, 0, None, None)
    for z in zs:
        L, S, ls, ss = axes_2d(mask[z], sy, sx)
        if L > best[1]:
            best = (int(z), L, S, ls, ss)
    z, L, S, ls, ss = best
    vox = mask.sum()
    vol = float(vox * sz * sy * sx)
    hu = ct_hu[mask]
    solid = mask & (ct_hu > SOLID_HU)
    # judge density on the core (edge voxels are blurred with air and look falsely 'ground-glass')
    core = ndi.binary_erosion(mask, np.ones((1, 3, 3)))
    if core.sum() < 0.3 * vox:
        core = mask
    solid_frac = float((core & (ct_hu > SOLID_HU)).sum() / max(core.sum(), 1))
    if solid_frac >= 0.8:
        typ, solid_mm = "Solid", None
    elif solid_frac < 0.1:
        typ, solid_mm = "Ground-glass", None
    else:
        typ = "Part-solid"
        solid_mm = round(max(axes_2d(solid[k], sy, sx)[0] for k in zs), 1)
    lobe, touches = None, False
    if lobe_map is not None:
        lab = lobe_map[mask]
        touches = bool((lab == 0).mean() > 0.2)
        lab = lab[lab > 0]
        if len(lab) == 0:  # nodule on the lung edge: take the nearest lobe
            sub = tuple(slice(max(int(np.min(i)) - 15, 0), int(np.max(i)) + 16) for i in np.nonzero(mask))
            lm = lobe_map[sub]
            if lm.any():
                _, idx = ndi.distance_transform_edt(lm == 0, sampling=spacing_zyx, return_indices=True)
                c = tuple(int(np.mean(i)) for i in np.nonzero(mask[sub]))
                lab = np.array([lm[idx[0][c], idx[1][c], idx[2][c]]])
        if len(lab):
            from .detect import LOBE_CODE
            lobe = LOBE_CODE[int(np.bincount(lab).argmax())]
    cz, cy, cx = [float(np.mean(i)) for i in np.nonzero(mask)]
    r = lambda x: None if x is None else round(float(x), 1)
    return dict(
        lobe=lobe, slice_index=z,
        slice_number=int(instance_numbers[z]) if instance_numbers is not None else z + 1,
        type=typ, long_axis_mm=r(L), short_axis_mm=r(S), mean_diameter_mm=r((L + S) / 2),
        solid_component_mm=solid_mm, volume_mm3=round(vol), mean_hu=round(float(hu.mean())),
        solid_fraction=round(solid_frac, 2), center_voxel=[round(cz, 1), round(cy, 1), round(cx, 1)],
        long_axis_px=[[round(float(p[0]), 1), round(float(p[1]), 1)] for p in ls] if ls else None,
        short_axis_px=[[round(float(p[0]), 1), round(float(p[1]), 1)] for p in ss] if ss else None,
        n_slices=int(len(zs)), touches_pleura=touches)


def components(nod_mask, prob=None, spacing_zyx=(1, 1, 1), min_long_mm=4.0, ct_hu=None, **kw):
    """Split the nodule mask into separate nodules, measure each, drop those < min_long_mm.
    Returns (list of dicts sorted by size, label map)."""
    lab, n = ndi.label(nod_mask, structure=np.ones((3, 3, 3)))
    out, keep = [], np.zeros_like(lab, dtype=np.int16)
    objs = ndi.find_objects(lab)
    k = 0
    for i, sl in enumerate(objs, 1):
        m = lab == i
        m_sl = m[sl]
        meas = measure_nodule(m_sl, ct_hu[sl], spacing_zyx, kw.get("lobe_map")[sl] if kw.get("lobe_map") is not None else None, None)
        if meas["long_axis_mm"] < min_long_mm:
            continue
        # translate crop-relative coordinates back to the full scan
        off = np.array([s.start for s in sl], float)
        meas["slice_index"] += int(off[0])
        inst = kw.get("instance_numbers")
        meas["slice_number"] = int(inst[meas["slice_index"]]) if inst is not None else meas["slice_index"] + 1
        meas["center_voxel"] = [round(c + o, 1) for c, o in zip(meas["center_voxel"], off)]
        for key in ("long_axis_px", "short_axis_px"):
            if meas[key]:
                meas[key] = [[p[0] + off[1], p[1] + off[2]] for p in meas[key]]
        meas["confidence"] = round(float(prob[m].max()), 2) if prob is not None else None
        meas["bbox"] = [[int(s.start), int(s.stop)] for s in sl]
        k += 1
        keep[m] = k
        out.append(meas)
    order = np.argsort([-o["volume_mm3"] for o in out])
    out = [out[i] for i in order]
    remap = np.zeros(k + 1, np.int16)
    for new, old in enumerate(order, 1):
        remap[old + 1] = new
        out[new - 1]["nodule_id"] = f"N{new}"
    return out, remap[keep]


def spur_mm(mask, spacing_zyx, open_mm=1.5):
    """How much of the long axis comes from a thin protrusion (e.g. a vessel tail stuck to the outline).
    = long axis of the outline minus long axis after an in-plane opening that removes parts thinner
    than ~2 x open_mm. Same slice-search as Daniel's measurement. 0 = no tail."""
    sz, sy, sx = spacing_zyx
    r = max(1, int(round(open_mm / sy)))
    yy, xx = np.ogrid[-r:r + 1, -r:r + 1]
    opened = ndi.binary_opening(mask, (yy ** 2 + xx ** 2 <= r * r)[None])
    if not opened.any():
        return 0.0
    L = lambda m: max(axes_2d(m[z], sy, sx)[0] for z in np.nonzero(m.any(axis=(1, 2)))[0])
    return round(max(0.0, L(mask) - L(opened)), 1)


In [ ]:
%%writefile /content/engine/match.py
"""Match nodules between a follow-up scan and its prior.

1. Line up the two scans: affine registration of the lungs (SimpleITK, Mattes mutual information,
   lungs only, 4 mm then 2 mm), started from the lung-mask centres. This absorbs breathing/position
   shifts, so slice numbers no longer need to agree.
2. Move each prior nodule centre into the follow-up scan, then pair nodules with the Hungarian
   method on centre distance. A pair is accepted only if the distance is below
   max(10 mm, 0.75 x the larger long axis).
Unpaired follow-up nodules are reported as 'new', unpaired prior ones as 'not found'.
"""
import numpy as np
import SimpleITK as sitk
from scipy.optimize import linear_sum_assignment


def register(fixed_ct, moving_ct, fixed_lungs, moving_lungs):
    """Returns a transform mapping FIXED physical points -> MOVING physical points."""
    f = sitk.Cast(fixed_ct, sitk.sitkFloat32)
    m = sitk.Cast(moving_ct, sitk.sitkFloat32)
    fl = sitk.GetImageFromArray((fixed_lungs > 0).astype(np.uint8)); fl.CopyInformation(fixed_ct)
    ml = sitk.GetImageFromArray((moving_lungs > 0).astype(np.uint8)); ml.CopyInformation(moving_ct)
    init = sitk.CenteredTransformInitializer(fl, ml, sitk.AffineTransform(3),
                                             sitk.CenteredTransformInitializerFilter.MOMENTS)
    fmask = sitk.BinaryDilate(fl, [3, 3, 3])
    R = sitk.ImageRegistrationMethod()
    R.SetMetricAsMattesMutualInformation(32)
    R.SetMetricFixedMask(fmask)
    R.SetMetricSamplingStrategy(R.RANDOM)
    R.SetMetricSamplingPercentage(0.05, seed=1)
    R.SetInterpolator(sitk.sitkLinear)
    R.SetOptimizerAsRegularStepGradientDescent(1.0, 1e-4, 150, relaxationFactor=0.6)
    R.SetOptimizerScalesFromPhysicalShift()
    R.SetShrinkFactorsPerLevel([4, 2])
    R.SetSmoothingSigmasPerLevel([2, 1])
    R.SetInitialTransform(init, inPlace=False)
    tx = R.Execute(f, m)
    return tx, R.GetMetricValue()


def vox_to_phys(img, zyx):
    return np.array(img.TransformContinuousIndexToPhysicalPoint([float(zyx[2]), float(zyx[1]), float(zyx[0])]))


def phys_to_vox(img, p):
    x, y, z = img.TransformPhysicalPointToContinuousIndex([float(v) for v in p])
    return [z, y, x]


def map_point(tx, fixed_img, moving_img, zyx_fixed):
    """A voxel in the fixed (current) scan -> the matching voxel in the moving (prior) scan."""
    return phys_to_vox(moving_img, tx.TransformPoint(tuple(vox_to_phys(fixed_img, zyx_fixed))))


def match(cur_nodules, prior_nodules, tx, cur_img, prior_img, min_gate_mm=10.0):
    inv = tx.GetInverse()  # prior physical -> current physical
    cur_p = np.array([vox_to_phys(cur_img, n["center_voxel"]) for n in cur_nodules]).reshape(-1, 3)
    pri_p = np.array([inv.TransformPoint(tuple(vox_to_phys(prior_img, n["center_voxel"])))
                      for n in prior_nodules]).reshape(-1, 3)
    pairs, used_c, used_p = [], set(), set()
    if len(cur_p) and len(pri_p):
        d = np.sqrt(((cur_p[:, None] - pri_p[None]) ** 2).sum(-1))
        r, c = linear_sum_assignment(d)
        for i, j in zip(r, c):
            gate = max(min_gate_mm, 0.75 * max(cur_nodules[i]["long_axis_mm"], prior_nodules[j]["long_axis_mm"]))
            if d[i, j] <= gate:
                pairs.append(dict(current=cur_nodules[i]["nodule_id"], prior=prior_nodules[j]["nodule_id"],
                                  distance_mm=round(float(d[i, j]), 1)))
                used_c.add(i); used_p.add(j)
    new = [n["nodule_id"] for i, n in enumerate(cur_nodules) if i not in used_c]
    gone = [n["nodule_id"] for j, n in enumerate(prior_nodules) if j not in used_p]
    return dict(pairs=pairs, new_in_current=new, not_found_in_current=gone)


In [ ]:
%%writefile /content/engine/outline.py
"""Outline refinement after detection.

The nodule model outlines the dense core well but tends to leave out the faint ground-glass rim
(seen on the first RIDER scan: long axis 9.8 mm vs expert 15.0 mm). Daniel measures the WHOLE nodule,
rim included, so we test refiners on top of the model outline:

- `halo`: grow the model outline into connected voxels denser than -750 HU, at most `band_mm` away,
  then cut thin vessel bridges. Cheap, CPU-friendly.
- MedSAM 2 (box prompt from the detection) is tested in the Colab exam; it needs a GPU.
"""
import numpy as np
from scipy import ndimage as ndi


def halo(mask, hu, spacing, band_mm=4.0, hu_min=-750, lungs=None):
    if not mask.any():
        return mask
    pad = np.ceil(band_mm / np.asarray(spacing)).astype(int) + 2
    zs, ys, xs = np.nonzero(mask)
    sl = tuple(slice(max(int(a.min()) - p, 0), int(a.max()) + p + 1) for a, p in zip((zs, ys, xs), pad))
    m = mask[sl]
    dist = ndi.distance_transform_edt(~m, sampling=spacing)
    allowed = (hu[sl] > hu_min) & (dist <= band_mm)
    if lungs is not None:
        allowed &= ndi.binary_dilation(lungs[sl] > 0, iterations=2)
    grown = m.copy()
    for _ in range(int(np.ceil(band_mm / min(spacing))) + 1):  # geodesic growth from the core
        nxt = ndi.binary_dilation(grown, np.ones((3, 3, 3))) & allowed | m
        if (nxt == grown).all():
            break
        grown = nxt
    # cut vessel bridges: in-plane opening of the added rim only, keep what touches the core
    rim = ndi.binary_opening(grown & ~m, np.ones((1, 3, 3)))
    out = m | rim
    lab, _ = ndi.label(out, np.ones((3, 3, 3)))
    keep = np.unique(lab[m]); keep = keep[keep > 0]
    out = np.isin(lab, keep)
    full = mask.copy(); full[sl] = out
    return full


In [ ]:
%%writefile /content/engine/pipeline.py
"""Full AI draft for one scan: lobes -> nodules -> measurements. Used by the API and the evaluation."""
import json, os, time
import numpy as np
import SimpleITK as sitk

from . import detect, measure

ENGINE_VERSION = "0.1.0"


def run_scan(ct_path, meta=None, cache_npz=None, min_long_mm=4.0, patch=None, step=0.5):
    """patch=None = the model's own patch (128x192x192). Smaller patches save memory but MISS nodules
    (tested 15 Oct: a 14 mm tumour went from found to probability 0.0), so do not shrink it.
    Returns (nodules list, label map (z,y,x) int16 with values = index in list + 1, lobes, prob, timings)."""
    t0 = time.time()
    ct = sitk.ReadImage(ct_path)
    hu = sitk.GetArrayFromImage(ct)
    sp = tuple(np.array(ct.GetSpacing())[::-1])
    tim = {}
    if cache_npz and os.path.exists(cache_npz):
        z = np.load(cache_npz)
        lobes, nod, prob = z["lobes"], z["nod"], z["prob"].astype(np.float32)
    else:
        t = time.time(); lobes = detect.lobes(ct_path); tim["lobes_s"] = round(time.time() - t, 1)
        t = time.time(); nod, prob = detect.nodules(ct_path, lobes, patch=patch, step=step)
        tim["nodules_s"] = round(time.time() - t, 1)
        if cache_npz:
            np.savez_compressed(cache_npz, lobes=lobes, nod=nod, prob=prob.astype(np.float16))
    inst = meta.get("instance_numbers") if meta else None
    t = time.time()
    nodules, labels = measure.components(nod, prob, sp, min_long_mm=min_long_mm, ct_hu=hu,
                                         lobe_map=lobes, instance_numbers=inst)
    for n in nodules:
        n["source"] = "ai"
        n["match_to_prior"] = None
    tim["measure_s"] = round(time.time() - t, 1)
    tim["total_s"] = round(time.time() - t0, 1)
    return nodules, labels, lobes, prob, tim


In [ ]:
%%writefile /content/engine/eval_detection.py
"""Exam for nodule FINDING, measuring and MATCHING on RIDER (test + retest, 15 minutes apart).

What RIDER can tell us, and what it can't:
- Each scan has ONE expert-outlined tumour. So we can check: did the tool find it? how close are its
  long/short axis and volume to the expert outline (measured the same way)?
- Other nodules in the scan are NOT marked. So we cannot count false alarms against ground truth.
  Proxy: the two scans are 15 minutes apart, so a real nodule should be found in BOTH. A candidate seen
  in only one of the two is suspicious (or a miss in the other). We report that "repeatability".
- Matching: the expert tumour in TEST and RETEST is the same lesion -> the pair must be matched.

Usage: python -m engine.eval_detection NII_DIR CACHE_DIR OUT_CSV [patient ...]
NII_DIR holds <pid>_<scan>_ct.nii.gz, _expert.nii.gz, _meta.json (made by to_nii.py).
"""
import json, os, sys, time
import numpy as np
import SimpleITK as sitk

from . import measure, match, outline
from .pipeline import run_scan


# "Too long because of a vessel tail" = long axis > expert + 2.3 mm (expert repeat noise) AND spur_mm > 2.
TAIL_RULE = dict(over_mm=2.3, spur_mm=2.0)

# Outline refiners compared on the found tumour: name -> fn(component_mask, hu, spacing, lobes) -> mask
REFINERS = {"halo": lambda c, hu, sp, lobes: outline.halo(c, hu, sp, 4.0, -750, lobes)}


def expert_measure(nii_dir, pid, scan, lobes):
    ct = sitk.ReadImage(f"{nii_dir}/{pid}_{scan}_ct.nii.gz")
    m = sitk.GetArrayFromImage(sitk.ReadImage(f"{nii_dir}/{pid}_{scan}_expert.nii.gz")) > 0
    meta = json.load(open(f"{nii_dir}/{pid}_{scan}_meta.json"))
    sp = tuple(np.array(ct.GetSpacing())[::-1])
    e = measure.measure_nodule(m, sitk.GetArrayFromImage(ct), sp, lobes, meta["instance_numbers"])
    return e, m, ct, meta


def eval_scan(nii_dir, cache_dir, pid, scan):
    meta = json.load(open(f"{nii_dir}/{pid}_{scan}_meta.json"))
    nods, labels, lobes, prob, tim = run_scan(f"{nii_dir}/{pid}_{scan}_ct.nii.gz", meta,
                                              cache_npz=f"{cache_dir}/{pid}_{scan}.npz")
    e, m, ct, _ = expert_measure(nii_dir, pid, scan, lobes)
    hits = [i for i in range(len(nods)) if (labels[m] == i + 1).any()]
    row = dict(patient=pid, scan=scan, n_candidates=len(nods), found=bool(hits),
               expert_long=e["long_axis_mm"], expert_short=e["short_axis_mm"], expert_vol=e["volume_mm3"],
               expert_lobe=e["lobe"], expert_type=e["type"], expert_slice=e["slice_number"])
    if hits:
        i = max(hits, key=lambda k: (labels[m] == k + 1).sum())
        n = nods[i]
        pm = labels == i + 1
        row.update(tool_id=n["nodule_id"], tool_long=n["long_axis_mm"], tool_short=n["short_axis_mm"],
                   tool_vol=n["volume_mm3"], tool_lobe=n["lobe"], tool_type=n["type"], tool_slice=n["slice_number"],
                   confidence=n["confidence"],
                   dice=round(2 * (pm & m).sum() / (pm.sum() + m.sum()), 3))
        hu = sitk.GetArrayFromImage(ct)
        sp = tuple(np.array(ct.GetSpacing())[::-1])
        sl = tuple(slice(max(a.min() - 5, 0), a.max() + 6) for a in np.nonzero(pm | m))
        row.update(tool_spur_mm=measure.spur_mm(pm[sl], sp), expert_spur_mm=measure.spur_mm(m[sl], sp))
        for name, fn in REFINERS.items():
            try:
                t = time.time()
                rm = fn(pm, hu, sp, lobes)
                r = measure.measure_nodule(rm, hu, sp)
                slr = tuple(slice(max(a.min() - 5, 0), a.max() + 6) for a in np.nonzero(rm | m))
                row[f"{name}_spur_mm"] = measure.spur_mm(rm[slr], sp)
                row.update({f"{name}_dice": round(2 * (rm & m).sum() / (rm.sum() + m.sum()), 3),
                            f"{name}_long": r["long_axis_mm"], f"{name}_short": r["short_axis_mm"],
                            f"{name}_vol": r["volume_mm3"], f"{name}_s": round(time.time() - t, 1)})
            except Exception as ex:
                row[f"{name}_error"] = str(ex)[:200]
    else:
        # how strong was the model's signal inside the expert tumour?
        row.update(max_prob_in_tumour=round(float(prob[m].max()), 3))
    row.update(tim)
    return row, nods, lobes, ct


def main():
    nii_dir, cache_dir, out_csv = sys.argv[1:4]
    os.makedirs(cache_dir, exist_ok=True)
    pids = sys.argv[4:] or sorted({f.split("_")[0] for f in os.listdir(nii_dir) if f.endswith("_meta.json")})
    import pandas as pd
    rows, pairs = [], []
    for pid in pids:
        per = {}
        for scan in ("TEST", "RETEST"):
            if not os.path.exists(f"{nii_dir}/{pid}_{scan}_meta.json"):
                continue
            t = time.time()
            row, nods, lobes, ct = eval_scan(nii_dir, cache_dir, pid, scan)
            rows.append(row); per[scan] = (row, nods, lobes, ct)
            print(pid, scan, {k: row.get(k) for k in ("found", "n_candidates", "expert_long", "tool_long", "dice")},
                  f"{time.time() - t:.0f}s", flush=True)
        if len(per) == 2:  # matching RETEST (current) -> TEST (prior)
            (rc, nc, lc, cc), (rp, np_, lp, cp) = per["RETEST"], per["TEST"]
            t = time.time()
            tx, metric = match.register(cc, cp, lc, lp)
            res = match.match(nc, np_, tx, cc, cp)
            ok = None
            if rc.get("tool_id") and rp.get("tool_id"):
                ok = {"current": rc["tool_id"], "prior": rp["tool_id"]} in [
                    {"current": p["current"], "prior": p["prior"]} for p in res["pairs"]]
            # where does the expert tumour centre land after registration? (registration error)
            e_c = np.array(match.vox_to_phys(cc, measure_center(nii_dir, pid, "RETEST")))
            e_p = np.array(match.vox_to_phys(cp, measure_center(nii_dir, pid, "TEST")))
            reg_err = float(np.linalg.norm(np.array(tx.TransformPoint(tuple(e_c))) - e_p))
            pairs.append(dict(patient=pid, tumour_matched=ok, registration_error_mm=round(reg_err, 1),
                              n_current=len(nc), n_prior=len(np_), n_pairs=len(res["pairs"]),
                              unpaired_current=len(res["new_in_current"]), unpaired_prior=len(res["not_found_in_current"]),
                              match_s=round(time.time() - t, 1)))
            print("  match:", pairs[-1], flush=True)
        pd.DataFrame(rows).to_csv(out_csv, index=False)
        pd.DataFrame(pairs).to_csv(out_csv.replace(".csv", "_matching.csv"), index=False)


def measure_center(nii_dir, pid, scan):
    m = sitk.GetArrayFromImage(sitk.ReadImage(f"{nii_dir}/{pid}_{scan}_expert.nii.gz")) > 0
    return [float(np.mean(i)) for i in np.nonzero(m)]


if __name__ == "__main__":
    main()


## 2. Download the scans + expert outlines from TCIA and convert them

In [ ]:
cases = [["RIDER-1129164940", "TEST", "1.2.276.0.7230010.3.1.3.8323329.2021.1554822307.24503", "1.3.6.1.4.1.9328.50.1.83304264089411327530730818890072724533"], ["RIDER-1129164940", "RETEST", "1.2.276.0.7230010.3.1.3.8323329.20780.1554842143.193341", "1.3.6.1.4.1.9328.50.1.48441840081578419409180519840073808100"], ["RIDER-1225316081", "TEST", "1.2.276.0.7230010.3.1.3.8323329.5678.1554820235.472211", "1.3.6.1.4.1.9328.50.1.136644362331225154777907863722323747874"], ["RIDER-1225316081", "RETEST", "1.2.276.0.7230010.3.1.3.8323329.7091.1554843537.973188", "1.3.6.1.4.1.9328.50.1.57875588819948681104622543575484490295"], ["RIDER-1286684383", "TEST", "1.2.276.0.7230010.3.1.3.8323329.18140.1554821167.472678", "1.3.6.1.4.1.9328.50.1.250276730034374548755125887754378560209"], ["RIDER-1286684383", "RETEST", "1.2.276.0.7230010.3.1.3.8323329.10537.1554843803.471316", "1.3.6.1.4.1.9328.50.1.54869681489123387741768260322440045297"], ["RIDER-1332496276", "TEST", "1.2.276.0.7230010.3.1.3.8323329.3432.1554822400.109841", "1.3.6.1.4.1.9328.50.1.42697596859477567872763647333745089432"], ["RIDER-1332496276", "RETEST", "1.2.276.0.7230010.3.1.3.8323329.18239.1554844358.919088", "1.3.6.1.4.1.9328.50.1.51182298817488926005730686623574691733"], ["RIDER-1344011628", "RETEST", "1.2.276.0.7230010.3.1.3.8323329.3483.1554843262.971275", "1.3.6.1.4.1.9328.50.1.66108458465186904026908357142054909583"], ["RIDER-1344011628", "TEST", "1.2.276.0.7230010.3.1.3.8323329.23231.1554821525.429266", "1.3.6.1.4.1.9328.50.1.70885448642275408073484639858923588464"], ["RIDER-1346917136", "TEST", "1.2.276.0.7230010.3.1.3.8323329.31653.1554822125.805873", "1.3.6.1.4.1.9328.50.1.298748100008818908989408006815250397386"], ["RIDER-1346917136", "RETEST", "1.2.276.0.7230010.3.1.3.8323329.16870.1554844272.214723", "1.3.6.1.4.1.9328.50.1.241687561050325650322607529919565760760"], ["RIDER-1500037140", "TEST", "1.2.276.0.7230010.3.1.3.8323329.6703.1554820324.285935", "1.3.6.1.4.1.9328.50.1.275712112233733729527949643353021665322"], ["RIDER-1500037140", "RETEST", "1.2.276.0.7230010.3.1.3.8323329.11604.1554843904.94647", "1.3.6.1.4.1.9328.50.1.228849430732466946515663085966276560854"], ["RIDER-1532432635", "TEST", "1.2.276.0.7230010.3.1.3.8323329.3151.1554820054.833742", "1.3.6.1.4.1.9328.50.1.244501235124778437799277943012383090930"], ["RIDER-1532432635", "RETEST", "1.2.276.0.7230010.3.1.3.8323329.23409.1554842387.560675", "1.3.6.1.4.1.9328.50.1.251497901416717723949588736705581667952"], ["RIDER-1760553574", "TEST", "1.2.276.0.7230010.3.1.3.8323329.4630.1554822493.511219", "1.3.6.1.4.1.9328.50.1.327399981350769389671009921076478636105"], ["RIDER-1760553574", "RETEST", "1.2.276.0.7230010.3.1.3.8323329.9236.1554843710.400783", "1.3.6.1.4.1.9328.50.1.217618293370677618119893300068426363153"], ["RIDER-1801720707", "TEST", "1.2.276.0.7230010.3.1.3.8323329.25780.1554821702.493501", "1.3.6.1.4.1.9328.50.1.269982481234503399884470277848651506493"], ["RIDER-1801720707", "RETEST", "1.2.276.0.7230010.3.1.3.8323329.26032.1554842575.927371", "1.3.6.1.4.1.9328.50.1.234211180786790460829763655261513482302"], ["RIDER-1825099523", "TEST", "1.2.276.0.7230010.3.1.3.8323329.15604.1554820995.783866", "1.3.6.1.4.1.9328.50.1.151205757722151678647268859397211234394"], ["RIDER-1825099523", "RETEST", "1.2.276.0.7230010.3.1.3.8323329.12950.1554844002.252553", "1.3.6.1.4.1.9328.50.1.277306877703927900763621250092700839081"], ["RIDER-1867199702", "TEST", "1.2.276.0.7230010.3.1.3.8323329.1705.1554819982.390085", "1.3.6.1.4.1.9328.50.1.159175208197607220484095971384563377994"], ["RIDER-1867199702", "RETEST", "1.2.276.0.7230010.3.1.3.8323329.2204.1554843187.867545", "1.3.6.1.4.1.9328.50.1.176840664438918409569682530893469371660"], ["RIDER-2016615262", "TEST", "1.2.276.0.7230010.3.1.3.8323329.4480.1554820147.296719", "1.3.6.1.4.1.9328.50.1.299673360876570070700703250582150584718"], ["RIDER-2016615262", "RETEST", "1.2.276.0.7230010.3.1.3.8323329.11318.1556878233.850552", "1.3.6.1.4.1.9328.50.1.211449458002306195716642985423685291150"], ["RIDER-2073083717", "RETEST", "1.2.276.0.7230010.3.1.3.8323329.15621.1554844195.560572", "1.3.6.1.4.1.9328.50.1.37531487280017801124765643732079524443"], ["RIDER-2073083717", "TEST", "1.2.276.0.7230010.3.1.3.8323329.29296.1554821950.541699", "1.3.6.1.4.1.9328.50.1.205940897977851990746556006144481623627"], ["RIDER-2151469008", "TEST", "1.2.276.0.7230010.3.1.3.8323329.16806.1554821083.229719", "1.3.6.1.4.1.9328.50.1.94933979665959337442766748630370536812"], ["RIDER-2151469008", "RETEST", "1.2.276.0.7230010.3.1.3.8323329.22045.1554842223.669661", "1.3.6.1.4.1.9328.50.1.284014020710168432486041624571005092281"], ["RIDER-2283289298", "RETEST", "1.2.276.0.7230010.3.1.3.8323329.14148.1554844092.679599", "1.3.6.1.4.1.9328.50.1.109288921990958367956324078058642091696"], ["RIDER-2357766186", "TEST", "1.2.276.0.7230010.3.1.3.8323329.14205.1554820902.237181", "1.3.6.1.4.1.9328.50.1.245630263591502535745452645381329674063"], ["RIDER-2357766186", "RETEST", "1.2.276.0.7230010.3.1.3.8323329.30717.1554842934.942943", "1.3.6.1.4.1.9328.50.1.160525591228102999616019562758104412505"], ["RIDER-2539508879", "TEST", "1.2.276.0.7230010.3.1.3.8323329.10399.1554820614.125440", "1.3.6.1.4.1.9328.50.1.158120423511327812902957049440486097165"], ["RIDER-2539508879", "RETEST", "1.2.276.0.7230010.3.1.3.8323329.19578.1554842047.820136", "1.3.6.1.4.1.9328.50.1.215800995114148697334431058444392949526"], ["RIDER-2541949645", "TEST", "1.2.276.0.7230010.3.1.3.8323329.20685.1554821352.112310", "1.3.6.1.4.1.9328.50.1.152572901056058406211409536989510187742"], ["RIDER-2541949645", "RETEST", "1.2.276.0.7230010.3.1.3.8323329.8070.1554843626.111330", "1.3.6.1.4.1.9328.50.1.84023861072193772859482594252670063850"], ["RIDER-2619750334", "TEST", "1.2.276.0.7230010.3.1.3.8323329.12891.1554820797.597451", "1.3.6.1.4.1.9328.50.1.307319453402340384126936296040122224868"], ["RIDER-2619750334", "RETEST", "1.2.276.0.7230010.3.1.3.8323329.26932.1554842656.958509", "1.3.6.1.4.1.9328.50.1.235386105589494260919744152317749907915"], ["RIDER-2655999012", "TEST", "1.2.276.0.7230010.3.1.3.8323329.364.1554819884.486584", "1.3.6.1.4.1.9328.50.1.112793496676844175431872842334447612042"], ["RIDER-2655999012", "RETEST", "1.2.276.0.7230010.3.1.3.8323329.18411.1554841949.273338", "1.3.6.1.4.1.9328.50.1.5804567971224333682982299447203570095"], ["RIDER-2669524182", "RETEST", "1.2.276.0.7230010.3.1.3.8323329.28168.1554842757.156486", "1.3.6.1.4.1.9328.50.1.269463014648045535201701159778694569334"], ["RIDER-2669524182", "TEST", "1.2.276.0.7230010.3.1.3.8323329.21953.1554821436.288515", "1.3.6.1.4.1.9328.50.1.164139597747545377343600469399472836968"], ["RIDER-2799584460", "TEST", "1.2.276.0.7230010.3.1.3.8323329.591.1554822215.562011", "1.3.6.1.4.1.9328.50.1.182276903811735605390981624025960600655"], ["RIDER-2799584460", "RETEST", "1.2.276.0.7230010.3.1.3.8323329.31907.1554843015.400512", "1.3.6.1.4.1.9328.50.1.292474823408043076724467505606424288322"], ["RIDER-3023568408", "TEST", "1.2.276.0.7230010.3.1.3.8323329.19345.1554821263.444952", "1.3.6.1.4.1.9328.50.1.177903673228104341213224928761701786693"], ["RIDER-3023568408", "RETEST", "1.2.276.0.7230010.3.1.3.8323329.24761.1554842482.122464", "1.3.6.1.4.1.9328.50.1.76808802982262101679502232663993705555"], ["RIDER-3115188676", "RETEST", "1.2.276.0.7230010.3.1.3.8323329.747.1554843102.753472", "1.3.6.1.4.1.9328.50.1.136792069117584747719409894247257396682"], ["RIDER-3115188676", "TEST", "1.2.276.0.7230010.3.1.3.8323329.7982.1554820417.354881", "1.3.6.1.4.1.9328.50.1.255647697326259514897843794470928560217"], ["RIDER-3152132495", "RETEST", "1.2.276.0.7230010.3.1.3.8323329.19480.1554844461.437970", "1.3.6.1.4.1.9328.50.1.334215469883390884437229836705492234507"], ["RIDER-3152132495", "TEST", "1.2.276.0.7230010.3.1.3.8323329.28010.1554821863.449566", "1.3.6.1.4.1.9328.50.1.116324576791139455082825091960790224867"], ["RIDER-3160137230", "RETEST", "1.2.276.0.7230010.3.1.3.8323329.4613.1554843349.990449", "1.3.6.1.4.1.9328.50.1.259840310625640684686783854863615824641"], ["RIDER-3160137230", "TEST", "1.2.276.0.7230010.3.1.3.8323329.26750.1554821783.648863", "1.3.6.1.4.1.9328.50.1.191985769478320383635122902894438276122"], ["RIDER-5195703382", "TEST", "1.2.276.0.7230010.3.1.3.8323329.11554.1554820706.857013", "1.3.6.1.4.1.9328.50.1.286170493057098735157197376891061719104"], ["RIDER-6324870494", "TEST", "1.2.276.0.7230010.3.1.3.8323329.24558.1554821613.346079", "1.3.6.1.4.1.9328.50.1.52136035328653293207079755735387340761"], ["RIDER-6324870494", "RETEST", "1.2.276.0.7230010.3.1.3.8323329.29519.1554842848.297024", "1.3.6.1.4.1.9328.50.1.286680119667247381026746579579720294047"], ["RIDER-8509201188", "TEST", "1.2.276.0.7230010.3.1.3.8323329.9112.1554820506.221187", "1.3.6.1.4.1.9328.50.1.232408293761209220974761841467324190361"], ["RIDER-9763310455", "TEST", "1.2.276.0.7230010.3.1.3.8323329.30503.1554822027.916239", "1.3.6.1.4.1.9328.50.1.153467821594768929811844380798337401116"], ["RIDER-9763310455", "RETEST", "1.2.276.0.7230010.3.1.3.8323329.5891.1554843444.209653", "1.3.6.1.4.1.9328.50.1.70949788529300490780576543460245043945"]][:N_CASES]

import os, io, zipfile, requests, glob, json, concurrent.futures as cf
import numpy as np, pydicom, SimpleITK as sitk
API = "https://services.cancerimagingarchive.net/nbia-api/services/v1/getImage?SeriesInstanceUID="
DATA, NII = "/content/rider", "/content/nii"
os.makedirs(NII, exist_ok=True)

def fetch(uid):
    out = f"{DATA}/{uid}"
    if os.path.isdir(out) and os.listdir(out): return uid, "cached"
    for _ in range(4):
        try:
            r = requests.get(API + uid, timeout=600); r.raise_for_status()
            os.makedirs(out, exist_ok=True); zipfile.ZipFile(io.BytesIO(r.content)).extractall(out); return uid, "ok"
        except Exception as e: err = e
    return uid, f"FAILED {err}"

uids = sorted({c[2] for c in cases} | {c[3] for c in cases})
with cf.ThreadPoolExecutor(6) as ex:
    for i, (u, st) in enumerate(ex.map(fetch, uids), 1):
        if st != "cached": print(i, len(uids), st)

def expert_mask(seg_uid, ct):
    ds = pydicom.dcmread(glob.glob(f"{DATA}/{seg_uid}/*.dcm")[0])
    no = next(s.SegmentNumber for s in ds.SegmentSequence if s.SegmentAlgorithmType == "MANUAL")
    fr = ds.pixel_array; fr = fr[None] if fr.ndim == 2 else fr
    nz = ct.GetSize()[2]; zs = np.array([ct.TransformIndexToPhysicalPoint((0, 0, k))[2] for k in range(nz)])
    m = np.zeros((nz, ct.GetSize()[1], ct.GetSize()[0]), bool)
    for i, fg in enumerate(ds.PerFrameFunctionalGroupsSequence):
        if fg.SegmentIdentificationSequence[0].ReferencedSegmentNumber != no: continue
        z = float(fg.PlanePositionSequence[0].ImagePositionPatient[2]); k = int(np.argmin(np.abs(zs - z)))
        if abs(zs[k] - z) < 0.5: m[k] |= fr[i] > 0
    return m

for pid, scan, seg_uid, ct_uid in cases:
    if os.path.exists(f"{NII}/{pid}_{scan}_meta.json"): continue
    d = f"{DATA}/{ct_uid}"; files = sitk.ImageSeriesReader.GetGDCMSeriesFileNames(d)
    r = sitk.ImageSeriesReader(); r.SetFileNames(files); ct = r.Execute()
    m = expert_mask(seg_uid, ct)
    sitk.WriteImage(ct, f"{NII}/{pid}_{scan}_ct.nii.gz", useCompression=True)
    mi = sitk.GetImageFromArray(m.astype(np.uint8)); mi.CopyInformation(ct)
    sitk.WriteImage(mi, f"{NII}/{pid}_{scan}_expert.nii.gz", useCompression=True)
    ds = pydicom.dcmread(files[0], stop_before_pixels=True)
    inst = [int(pydicom.dcmread(f, stop_before_pixels=True).InstanceNumber) for f in files]
    json.dump(dict(patient=pid, scan=scan, series_number=int(ds.SeriesNumber), slice_thickness=float(ds.SliceThickness),
                   study_date=str(ds.StudyDate), instance_numbers=inst, spacing=ct.GetSpacing(), size=ct.GetSize()),
              open(f"{NII}/{pid}_{scan}_meta.json", "w"))
print("converted:", len(glob.glob(f"{NII}/*_meta.json")))

## 3. MedSAM 2 (tested as an outline refiner on top of the detection)

In [ ]:
# MedSAM 2 as an outline refiner: box from the DETECTED nodule (enlarged), on its largest slice
%cd /content
!git clone -q --depth 1 https://github.com/bowang-lab/MedSAM2.git
%cd /content/MedSAM2
!SAM2_BUILD_CUDA=0 pip install -q --no-deps -e . && pip install -q hydra-core iopath
!mkdir -p checkpoints && wget -q -nc -P checkpoints https://huggingface.co/wanglab/MedSAM2/resolve/main/MedSAM2_CTLesion.pt
%cd /content
import sys, numpy as np, torch
sys.path.insert(0, "/content/MedSAM2")
from PIL import Image
from skimage import measure as skm
from sam2.build_sam import build_sam2_video_predictor_npz
WINDOW = (-1350, 150)
AMP = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
MEAN = torch.tensor((0.485, 0.456, 0.406))[:, None, None].cuda(); STD = torch.tensor((0.229, 0.224, 0.225))[:, None, None].cuda()
PRED = build_sam2_video_predictor_npz("configs/sam2.1_hiera_t512.yaml", "/content/MedSAM2/checkpoints/MedSAM2_CTLesion.pt")

def to_frames(vol_u8, size=512):
    out = np.zeros((len(vol_u8), 3, size, size), np.float32)
    for i, s in enumerate(vol_u8):
        out[i] = np.array(Image.fromarray(s).convert("RGB").resize((size, size))).transpose(2, 0, 1)
    return (torch.from_numpy(out / 255.0).cuda() - MEAN) / STD

def medsam2_box(ct_arr, z, box, half=60):
    z0, z1 = max(0, z - half), min(len(ct_arr), z + half + 1)
    sub = np.clip(ct_arr[z0:z1], *WINDOW).astype(np.float32)
    sub = np.uint8((sub - sub.min()) / max(1e-6, sub.max() - sub.min()) * 255)
    H, W = sub.shape[1:]; y0, y1, x0, x1 = box
    seg = np.zeros(sub.shape, np.uint8); frames = to_frames(sub)
    with torch.inference_mode(), torch.autocast("cuda", dtype=AMP):
        for rev in (False, True):
            st = PRED.init_state(frames, H, W)
            PRED.add_new_points_or_box(inference_state=st, frame_idx=z - z0, obj_id=1, box=np.array([x0, y0, x1 - 1, y1 - 1]))
            for fi, _, lg in PRED.propagate_in_video(st, reverse=rev):
                seg[fi, (lg[0] > 0.0).cpu().numpy()[0]] = 1
            PRED.reset_state(st)
    out = np.zeros(ct_arr.shape, bool)
    if seg.any():
        lab = skm.label(seg); out[z0:z1] = lab == np.argmax(np.bincount(lab.flat)[1:]) + 1
    return out

def medsam2_refiner(c, hu, sp, lobes, grow=0.25):
    z = int(np.argmax(c.sum(axis=(1, 2)))); ys, xs = np.nonzero(c[z])
    h, w = ys.max() - ys.min() + 1, xs.max() - xs.min() + 1
    py, px = max(3, int(grow * h)), max(3, int(grow * w))
    box = (max(ys.min() - py, 0), ys.max() + py + 1, max(xs.min() - px, 0), xs.max() + px + 1)
    return medsam2_box(hu, z, box)
print("MedSAM 2 ready")

## 4. Run the exam (GPU: full model patch, ~20–40 s per scan incl. measuring)

In [ ]:
import sys; sys.path.insert(0, "/content")
import engine.eval_detection as ev
ev.REFINERS["medsam2"] = medsam2_refiner
pids = sorted({c[0] for c in cases})
sys.argv = ["x", "/content/nii", "/content/cache", "/content/finding_results.csv"] + pids
ev.main()

## 5. Summary

In [ ]:
import pandas as pd
r = pd.read_csv("/content/finding_results.csv"); mt = pd.read_csv("/content/finding_results_matching.csv")
f = r[r.found]
print(f"Expert tumour found: {r.found.sum()}/{len(r)} scans")
print(f"Long axis  |tool - expert| median {(f.tool_long - f.expert_long).abs().median():.1f} mm")
print(f"Short axis |tool - expert| median {(f.tool_short - f.expert_short).abs().median():.1f} mm")
print(f"Volume error median {((f.tool_vol - f.expert_vol).abs() / f.expert_vol * 100).median():.0f} %")
print(f"Dice median {f.dice.median():.3f}")
for v in ["halo", "medsam2"]:
    if f"{v}_dice" in f:
        print(f"  {v}: Dice {f[v+'_dice'].median():.3f} | long err {(f[v+'_long']-f.expert_long).abs().median():.1f} mm | short err {(f[v+'_short']-f.expert_short).abs().median():.1f} mm | vol err {((f[v+'_vol']-f.expert_vol).abs()/f.expert_vol*100).median():.0f} %")
print("By expert type:"); print(r.groupby("expert_type").agg(scans=("found", "size"), found=("found", "sum")))
print(f"Lobe agrees with lobe of expert outline: {(f.tool_lobe == f.expert_lobe).mean()*100:.0f} %")
print(f"Candidates per scan (>=4 mm): median {r.n_candidates.median():.0f}, max {r.n_candidates.max()}")
print(f"Tumour paired correctly test<->retest: {mt.tumour_matched.sum()}/{mt.tumour_matched.notna().sum()}")
print(f"Registration error at the tumour: median {mt.registration_error_mm.median():.1f} mm")
print(f"Other candidates seen in both scans: {mt.n_pairs.sum()}/{mt.n_current.sum()} (retest side)")
print(f"GPU seconds per scan (find): median {r.nodules_s.median():.0f}")
r[~r.found]

## 6. Download results (send both files back)

In [ ]:
!cd /content && zip -q -r finding_results.zip finding_results*.csv && zip -q -r -0 engine_cache.zip cache
from google.colab import files
files.download("/content/finding_results.zip"); files.download("/content/engine_cache.zip")